# Section 7 — Adaptive RK23 and Comparison of ODE Methods

[← RK4](06_rk4.ipynb) · [Module 3](README.md)

This final notebook introduces the adaptive **RK2(3)** method used in Leon Q. Brin's _Tea Time Numerical Analysis_, Section 6.5, and then compares the numerical methods developed throughout Module 3.

> **Terminology note.** "RK23" is not a unique algorithm. Brin's companion `rk23.m` uses Heun's three-stage third-order method together with a second/third-order error estimate. SciPy's `solve_ivp(method="RK23")` uses a different embedded pair (Bogacki–Shampine). Here we implement **Brin's RK23** so the notebook matches the course text.

### Learning goals
- understand why adaptive step sizes are useful;
- implement Brin's RK23 controller in Python;
- visualize accepted step sizes;
- compare Euler, Taylor, RK2, RK3, RK4, and RK23 on one IVP.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Brin's adaptive RK23 pair

For the current point \((t_n,w_n)\), compute

\[
\begin{aligned}
k_1 &= f(t_n,w_n),\\
k_2 &= f\left(t_n+\frac h3,w_n+\frac h3k_1\right),\\
k_3 &= f\left(t_n+\frac{2h}{3},w_n+\frac{2h}{3}k_2\right).
\end{aligned}
\]

The third-order accepted value is

\[
w_{n+1}=w_n+\frac h4(k_1+3k_3),
\]

and Brin's local error estimate is

\[
R=\left|\frac h4(k_1-2k_2+k_3)\right|.
\]

If \(R\le \varepsilon\), accept the step. The next step size is chosen using

\[
q=0.9\left(\frac{\varepsilon}{R}\right)^{1/3},
\]

with \(q\) restricted to a reasonable range.

The Python version below follows Brin's algorithm closely. One small safety change is used: a step shortened to land exactly at the final time is still required to satisfy the tolerance before it is accepted.

In [ ]:
def rk23_brin(f, a, ya, b, h0=0.1, tol=1e-5, max_steps=10000):
    t_values = [float(a)]
    y_values = [float(ya)]
    h_values = []
    error_estimates = []
    rejected = 0
    evals = 0
    h = float(h0)

    while t_values[-1] < b and len(h_values) + rejected < max_steps:
        t = t_values[-1]
        y = y_values[-1]

        h = min(h, b - t)

        k1 = f(t, y)
        k2 = f(t + h/3, y + (h/3)*k1)
        k3 = f(t + 2*h/3, y + (2*h/3)*k2)
        evals += 3

        signed_estimate = (h/4)*(k1 - 2*k2 + k3)
        err = abs(signed_estimate)

        if err <= tol or h <= np.finfo(float).eps * max(1.0, abs(t)):
            y_new = y + (h/4)*(k1 + 3*k3)
            t_values.append(t + h)
            y_values.append(y_new)
            h_values.append(h)
            error_estimates.append(err)
        else:
            rejected += 1

        if err == 0:
            q = 5.0
        else:
            q = 0.9*(tol/err)**(1/3)

        q = min(5.0, max(0.1, q))
        h *= q

    if t_values[-1] < b:
        raise RuntimeError("Maximum number of trial steps reached.")

    return {
        "t": np.array(t_values),
        "y": np.array(y_values),
        "h": np.array(h_values),
        "err_est": np.array(error_estimates),
        "evals": evals,
        "rejected": rejected,
    }

## Adaptive example

Use the same course IVP

\[
y'=t e^{3t}-2y,\qquad y(0)=0,\qquad 0\le t\le1.
\]

The rapidly increasing \(e^{3t}\) term makes this a useful illustration: the solver should generally become more cautious as the solution changes more rapidly.

In [ ]:
def f(t, y):
    return t*np.exp(3*t) - 2*y

def exact(t):
    return np.exp(3*t)*(t/5 - 1/25) + np.exp(-2*t)/25

rk23 = rk23_brin(f, 0.0, 0.0, 1.0, h0=0.25, tol=1e-5)

print(f"Accepted steps: {len(rk23['h'])}")
print(f"Rejected trial steps: {rk23['rejected']}")
print(f"Function evaluations: {rk23['evals']}")
print(f"Approximation y(1): {rk23['y'][-1]:.10f}")
print(f"Exact y(1):         {exact(1.0):.10f}")
print(f"Absolute error:     {abs(rk23['y'][-1]-exact(1.0)):.3e}")

In [ ]:
t_fine = np.linspace(0, 1, 500)

fig, ax = plt.subplots(figsize=(9,5))
ax.plot(t_fine, exact(t_fine), linewidth=2, label="Exact")
ax.plot(rk23["t"], rk23["y"], marker="o", label="Brin RK23")
ax.set_xlabel("t")
ax.set_ylabel("y")
ax.set_title("Adaptive RK23 solution")
ax.grid(True, alpha=0.25)
ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9,4))
ax.step(rk23["t"][:-1], rk23["h"], where="post")
ax.set_xlabel("t")
ax.set_ylabel("accepted step size h")
ax.set_title("How RK23 adapts the step size")
ax.grid(True, alpha=0.25)
plt.show()

# Final comparison: all methods

To compare the methods fairly, the fixed-step methods below all use \(h=0.125\).  
RK23 is adaptive and instead uses a tolerance of \(10^{-5}\).

The table includes:

1. Euler;
2. Taylor order 2;
3. Taylor order 3;
4. Improved Euler (RK2);
5. Midpoint RK2;
6. Heun RK3;
7. Classical RK4;
8. Brin RK23.

A direct cost comparison between Taylor methods and Runge–Kutta methods is imperfect because Taylor methods require derivative formulas in addition to evaluations of \(f\). For the RK methods, the number of \(f\)-evaluations is shown explicitly.

In [ ]:
def y1(t, y):
    return f(t, y)

def y2(t, y):
    return np.exp(3*t)*(1+t) + 4*y

def y3(t, y):
    return np.exp(3*t)*(4+7*t) - 8*y

def euler_step(f, t, y, h):
    return y + h*f(t, y)

def taylor2_step(f_unused, t, y, h):
    return y + h*y1(t,y) + (h**2/2)*y2(t,y)

def taylor3_step(f_unused, t, y, h):
    return y + h*y1(t,y) + (h**2/2)*y2(t,y) + (h**3/6)*y3(t,y)

def rk2_improved_step(f, t, y, h):
    k1 = f(t, y)
    k2 = f(t+h, y+h*k1)
    return y + 0.5*h*(k1+k2)

def rk2_midpoint_step(f, t, y, h):
    k1 = f(t, y)
    k2 = f(t+h/2, y+(h/2)*k1)
    return y + h*k2

def rk3_heun_step(f, t, y, h):
    k1 = f(t, y)
    k2 = f(t+h/3, y+(h/3)*k1)
    k3 = f(t+2*h/3, y+(2*h/3)*k2)
    return y + (h/4)*(k1+3*k3)

def rk4_step(f, t, y, h):
    k1 = f(t, y)
    k2 = f(t+h/2, y+(h/2)*k1)
    k3 = f(t+h/2, y+(h/2)*k2)
    k4 = f(t+h, y+h*k3)
    return y + (h/6)*(k1+2*k2+2*k3+k4)

def fixed_step_solver(step_function, f, t0, y0, tf, h):
    n = round((tf-t0)/h)
    t = np.empty(n+1)
    y = np.empty(n+1)
    t[0], y[0] = t0, y0
    for i in range(n):
        y[i+1] = step_function(f, t[i], y[i], h)
        t[i+1] = t[i] + h
    return t, y

In [ ]:
h = 0.125
fixed_methods = [
    ("Euler", euler_step, 1, 1),
    ("Taylor 2", taylor2_step, 2, None),
    ("Taylor 3", taylor3_step, 3, None),
    ("Improved Euler RK2", rk2_improved_step, 2, 2),
    ("Midpoint RK2", rk2_midpoint_step, 2, 2),
    ("Heun RK3", rk3_heun_step, 3, 3),
    ("Classical RK4", rk4_step, 4, 4),
]

rows = []
curves = {}

for name, step, order, f_per_step in fixed_methods:
    t, y = fixed_step_solver(step, f, 0.0, 0.0, 1.0, h)
    curves[name] = (t, y)
    f_evals = "—" if f_per_step is None else str((len(t)-1)*f_per_step)
    rows.append((name, str(order), str(len(t)-1), f_evals, y[-1], abs(y[-1]-exact(1.0))))

curves["Brin RK23"] = (rk23["t"], rk23["y"])
rows.append(("Brin RK23", "3 (adaptive)", str(len(rk23["h"])), str(rk23["evals"]), rk23["y"][-1], abs(rk23["y"][-1]-exact(1.0))))

print(f"{'Method':22s} {'Order':>12s} {'Steps':>7s} {'f evals':>9s} {'y(1)':>14s} {'Abs. error':>14s}")
print("-"*86)
for row in rows:
    name, order, steps, evals, approx, err = row
    print(f"{name:22s} {order:>12s} {steps:>7s} {evals:>9s} {approx:14.9f} {err:14.6e}")

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))
ax.plot(t_fine, exact(t_fine), linewidth=3, label="Exact")
for name, (t, y) in curves.items():
    ax.plot(t, y, marker=".", label=name)
ax.set_xlabel("t")
ax.set_ylabel("y")
ax.set_title("All Module 3 ODE methods on the same IVP")
ax.grid(True, alpha=0.25)
ax.legend(fontsize=8)
plt.show()

## What the comparison shows

The methods form a clear progression:

- **Euler** is inexpensive but only first order.
- **Taylor 2 and Taylor 3** gain accuracy by computing higher derivatives.
- **RK2 and RK3** gain accuracy using additional slope evaluations instead of analytic derivatives.
- **RK4** gives high fixed-step accuracy with four slope evaluations per step.
- **RK23** adds a second idea: the step size itself becomes part of the numerical algorithm.

That last point is central to modern ODE solvers. The goal is no longer merely to choose a good formula; the solver also decides **where it needs more resolution**.

### References used for this sequence
- Leon Q. Brin, _Tea Time Numerical Analysis_, Chapter 6, especially Sections 6.3–6.5 and the companion `rk23.m` implementation.
- Richard L. Burden and J. Douglas Faires, _Numerical Analysis_, Runge–Kutta and adaptive step-size material.

[↑ Module 3](README.md)